In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:


# ===================== 2. IMPORT =====================
import pandas as pd
import os

# ===================== 3. DATASET PATH =====================
# 🔥 apna file path yahan change karo
data_path = "/content/drive/MyDrive/archive/House Price Prediction Dataset.csv"

# ===================== 4. LOAD DATA =====================
df = pd.read_csv(data_path)

# ===================== 5. BASIC INFO =====================
print("\n📌 FIRST 5 ROWS:")
print(df.head())

print("\n📌 LAST 5 ROWS:")
print(df.tail())

print("\n📌 DATASET SHAPE (Rows, Columns):")
print(df.shape)

print("\n📌 COLUMN NAMES:")
print(df.columns.tolist())

# ===================== 6. DATA TYPES =====================
print("\n📌 DATA TYPES:")
print(df.dtypes)

# ===================== 7. NULL VALUES =====================
print("\n📌 MISSING VALUES:")
print(df.isnull().sum())

# ===================== 8. STATISTICS =====================
print("\n📌 STATISTICS:")
print(df.describe())

# ===================== 9. UNIQUE VALUES (Categorical Check) =====================
print("\n📌 UNIQUE VALUES PER COLUMN:")
for col in df.columns:
    print(f"{col}: {df[col].nunique()} unique values")

# ===================== 10. SAMPLE VALUES =====================
print("\n📌 SAMPLE VALUES:")
for col in df.columns:
    print(f"{col} -> {df[col].unique()[:5]}")

# ===================== DONE =====================
print("\n✅ DATASET CHECK COMPLETE")


📌 FIRST 5 ROWS:
   Id  Area  Bedrooms  Bathrooms  Floors  YearBuilt  Location  Condition  \
0   1  1360         5          4       3       1970  Downtown  Excellent   
1   2  4272         5          4       3       1958  Downtown  Excellent   
2   3  3592         2          2       3       1938  Downtown       Good   
3   4   966         4          2       2       1902  Suburban       Fair   
4   5  4926         1          4       2       1975  Downtown       Fair   

  Garage   Price  
0     No  149919  
1     No  424998  
2     No  266746  
3    Yes  244020  
4    Yes  636056  

📌 LAST 5 ROWS:
        Id  Area  Bedrooms  Bathrooms  Floors  YearBuilt  Location  Condition  \
1995  1996  4994         5          4       3       1923  Suburban       Poor   
1996  1997  3046         5          2       1       2019  Suburban       Poor   
1997  1998  1062         5          1       2       1903     Rural       Poor   
1998  1999  4062         3          1       2       1936     Urban  Exce

In [5]:
# ===================== 1. DRIVE =====================
from google.colab import drive
drive.mount('/content/drive')

# ===================== 2. IMPORTS =====================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# ===================== 3. SAVE DIR =====================
model_name = "HousePrice_Advanced_" + datetime.now().strftime("%Y%m%d_%H%M%S")
save_dir = f"/content/drive/MyDrive/HousePrice_Results/{model_name}"
os.makedirs(save_dir, exist_ok=True)

print("Saving to:", save_dir)

# ===================== 4. LOAD DATA =====================
data_path = "/content/drive/MyDrive/archive/House Price Prediction Dataset.csv"
df = pd.read_csv(data_path)

# ===================== 5. FEATURE ENGINEERING =====================

# Remove ID
df = df.drop("Id", axis=1)

# House Age (IMPORTANT FEATURE)
df["HouseAge"] = 2026 - df["YearBuilt"]
df = df.drop("YearBuilt", axis=1)

# Outlier removal (PRICE)
df = df[df["Price"] < df["Price"].quantile(0.99)]

# One-hot encoding
df = pd.get_dummies(df, drop_first=True)

# ===================== 6. SPLIT =====================
X = df.drop("Price", axis=1)
y = df["Price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ===================== 7. MODELS (TUNED) =====================
models = {
    "RandomForest": RandomForestRegressor(
        n_estimators=400,
        max_depth=15,
        random_state=42
    ),
    "GradientBoosting": GradientBoostingRegressor(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )
}

results = {}

# ===================== 8. TRAIN =====================
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))

    results[name] = {"model": model, "MAE": mae, "RMSE": rmse}

    print(f"\n{name}")
    print("MAE :", mae)
    print("RMSE:", rmse)

# ===================== 9. BEST MODEL =====================
best_model_name = min(results, key=lambda x: results[x]["RMSE"])
best_model = results[best_model_name]["model"]

print("\n🏆 BEST MODEL:", best_model_name)

# ===================== 10. PREDICTION =====================
y_pred = best_model.predict(X_test)

# ===================== 11. SAVE RESULTS =====================
with open(os.path.join(save_dir, "results.txt"), "w") as f:
    for name in results:
        f.write(f"{name} -> MAE: {results[name]['MAE']}, RMSE: {results[name]['RMSE']}\n")
    f.write(f"\nBest Model: {best_model_name}")

# ===================== 12. GRAPH =====================
plt.figure()
plt.scatter(y_test, y_pred)
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title(f"Actual vs Predicted ({best_model_name})")
plt.savefig(os.path.join(save_dir, "actual_vs_pred.png"))
plt.close()

# ===================== 13. ERROR PLOT =====================
errors = y_test - y_pred

plt.figure()
plt.hist(errors, bins=30)
plt.title("Error Distribution")
plt.savefig(os.path.join(save_dir, "error_distribution.png"))
plt.close()

# ===================== 14. FEATURE IMPORTANCE =====================
importances = best_model.feature_importances_

plt.figure()
plt.barh(X.columns, importances)
plt.title("Feature Importance")
plt.savefig(os.path.join(save_dir, "feature_importance.png"))
plt.close()

# ===================== 15. SAVE MODEL =====================
joblib.dump(best_model, os.path.join(save_dir, best_model_name + ".pkl"))

# ===================== DONE =====================
print("\n✅ ADVANCED MODEL TRAINING COMPLETE")
print("Saved in:", save_dir)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saving to: /content/drive/MyDrive/HousePrice_Results/HousePrice_Advanced_20260417_110335

RandomForest
MAE : 237732.85008129885
RMSE: 276942.6276479786

GradientBoosting
MAE : 242600.01269715233
RMSE: 284650.9545980096

🏆 BEST MODEL: RandomForest

✅ ADVANCED MODEL TRAINING COMPLETE
Saved in: /content/drive/MyDrive/HousePrice_Results/HousePrice_Advanced_20260417_110335
